In [ ]:
import sqlite3
import shutil
from datasets import load_dataset
!pip install datasets esprima
# Load the dataset from Hugging Face





     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for esprima: filename=esprima-4.0.1-py3-none-any.whl size=62243 sha256=8dcf6dc14457266127355a954be0d47e1270531ddf2349b76c3a877e8c218a67
  Stored in directory: /root/.cache/pip/wheels/29/07/98/c81693d054f3f6283dd60ae0e41198be75372128b2fcc198b6
Successfully built esprima


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/212M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/428M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/580M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12987 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12987 [00:00<?, ? examples/s]

Number of JS snippets: 883
First CVE ID: CVE-2020-7763


In [2]:
ds = load_dataset("hitoshura25/cvefixes", split="train")

# Filter for JavaScript snippets
js_data = ds.filter(lambda x: x['language'] == 'JavaScript')

# Inspect the first entry
print(f"Number of JS snippets: {len(js_data)}")
print(f"First CVE ID: {js_data[0]['cve_id']}")


Number of JS snippets: 883
First CVE ID: CVE-2020-7763


In [5]:
db_path = 'processed_cvefixes.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
import json

# Create the table schema based on your sample structure
cursor.execute("""
CREATE TABLE IF NOT EXISTS processed_samples (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    cwe_id TEXT,
    code TEXT,
    target INTEGER,
    language TEXT,
    dataset TEXT,
    idx INTEGER,
    source_cve TEXT
)
""")

# 2. Load the Hugging Face Dataset
print("Loading dataset from Hugging Face...")
ds = load_dataset("hitoshura25/cvefixes", split="train")

# 3. Process and Insert Data
print("Processing and inserting records...")
idx = 200001
batch_size = 500
buffer = []

for row in ds:
    # Filter for JavaScript and ensure code exists
    if row['language'] == 'JavaScript' and row['vulnerable_code'] and row['fixed_code']:
        
        cve_id = row['cve_id']
        # Handle the CWE_ID list logic
        cwe_list = [row['cwe_id']] if row['cwe_id'] else ["UNKNOWN"]
        cwe_json = json.dumps(cwe_list) # Store list as JSON string

        # Prepare Vulnerable (target 1) and Fixed (target 0) entries
        buffer.append((cwe_json, row['vulnerable_code'], 1, "javascript", "cvefixes", idx, cve_id))
        idx += 1
        buffer.append((cwe_json, row['fixed_code'], 0, "javascript", "cvefixes", idx, cve_id))
        idx += 1

        # Commit in batches for speed
        if len(buffer) >= batch_size:
            cursor.executemany("""
                INSERT INTO processed_samples (cwe_id, code, target, language, dataset, idx, source_cve)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, buffer)
            buffer = []

# Insert any remaining records
if buffer:
    cursor.executemany("""
        INSERT INTO processed_samples (cwe_id, code, target, language, dataset, idx, source_cve)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    """, buffer)

conn.commit()
print(f"Database sync complete. Total samples: {idx - 200001}")
conn.close()

Loading dataset from Hugging Face...
Processing and inserting records...
Database sync complete. Total samples: 1572


In [6]:
conn = sqlite3.connect('processed_cvefixes.db')
res = conn.execute("SELECT code, target FROM processed_samples LIMIT 2").fetchall()
for code, target in res:
    print(f"Target: {target} | Code snippet length: {len(code)}")
conn.close()

Target: 1 | Code snippet length: 747
Target: 0 | Code snippet length: 1396


In [13]:
import os
from google.colab import drive
drive.mount('/content/drive')
local_db_path = '/content/processed_cvefixes.db'
drive_folder = '/content/drive/MyDrive/' # Change this to your folder
drive_db_path = os.path.join(drive_folder, '')
shutil.move(local_db_path, drive_db_path)

Mounted at /content/drive


'/content/drive/MyDrive/processed_cvefixes.db'